In [1]:
"""
Run this locally: pip install yfinance
Then: python fetch_market_data.py [TICKER]  (default TICKER = SPY)

Pulls the four inputs the pricing model needs:
  - S0: current spot price
  - sigma: at-the-money implied volatility, from the options expiry
           closest to ~1 year out (falls back to nearest available
           expiry and prints a warning if none are that far out)
  - r: risk-free rate, TERM-MATCHED to the option's actual maturity via
       linear interpolation across the Treasury curve -- not just the
       13-week bill rate, which would be a real mismatch for a ~1-year
       option if the curve isn't flat
  - q: trailing dividend yield (continuous-yield approximation)

Writes market_params.json in the current directory, which
run_experiment.py auto-detects and uses instead of its synthetic
placeholder defaults.
"""

import sys
import json
from datetime import datetime

import numpy as np
import yfinance as yf


def get_spot(ticker: str) -> float:
    t = yf.Ticker(ticker)
    hist = t.history(period="5d")
    if hist.empty:
        raise RuntimeError(f"No price history returned for {ticker}")
    return float(hist["Close"].iloc[-1])


def get_atm_implied_vol(ticker: str, spot: float, target_days: int = 365):
    """
    Find the options expiry closest to target_days out, then take the
    implied vol of the call whose strike is closest to the current spot
    (i.e. the ATM call). Returns (sigma, expiry_used, days_to_expiry, strike).
    """
    t = yf.Ticker(ticker)
    expiries = t.options
    if not expiries:
        raise RuntimeError(f"No options chain available for {ticker}")

    today = datetime.now()
    expiry_days = [(e, (datetime.strptime(e, "%Y-%m-%d") - today).days) for e in expiries]
    best_expiry, best_days = min(expiry_days, key=lambda x: abs(x[1] - target_days))

    chain = t.option_chain(best_expiry)
    calls = chain.calls
    if calls.empty:
        raise RuntimeError(f"No call quotes for {ticker} expiry {best_expiry}")

    calls = calls.copy()
    calls["strike_dist"] = (calls["strike"] - spot).abs()
    atm_row = calls.sort_values("strike_dist").iloc[0]

    iv = float(atm_row["impliedVolatility"])
    return iv, best_expiry, best_days, float(atm_row["strike"])


def get_risk_free_rate(target_years: float) -> float:
    """
    Term-matched risk-free rate via linear interpolation between Treasury
    yield curve points. Using the 13-week bill to discount a ~1-year
    option is a real mismatch if the curve isn't flat -- this fixes that.

    Free Yahoo tickers available: ^IRX (13-week bill, ~0.25y), ^FVX
    (5-year note), ^TNX (10-year note), ^TYX (30-year bond). There's no
    clean 1-2y point this way, so for target maturities between 0.25y
    and 5y this linearly interpolates between ^IRX and ^FVX. That's a
    real approximation -- the actual curve isn't linear there, and
    short-to-intermediate curvature is often where the curve bends the
    most -- but it's a better estimate than pinning everything to the
    3-month rate regardless of option maturity.
    """
    tenors = {"^IRX": 0.25, "^FVX": 5.0, "^TNX": 10.0, "^TYX": 30.0}
    points = []
    for ticker, years in tenors.items():
        try:
            hist = yf.Ticker(ticker).history(period="5d")
            if not hist.empty:
                points.append((years, float(hist["Close"].iloc[-1]) / 100.0))
        except Exception:
            continue

    if not points:
        raise RuntimeError("Could not fetch any Treasury yield curve points")

    points.sort()
    years_arr = [p[0] for p in points]
    rates_arr = [p[1] for p in points]

    if target_years <= years_arr[0]:
        return rates_arr[0]
    if target_years >= years_arr[-1]:
        return rates_arr[-1]
    return float(np.interp(target_years, years_arr, rates_arr))


def get_dividend_yield(ticker: str) -> float:
    """Trailing dividend yield as a decimal (e.g. 0.013 for 1.3%).
    Falls back to 0.0 with a warning if unavailable -- some tickers
    (individual growth stocks, non-dividend payers) legitimately have
    none, and yfinance's info dict is not always populated or consistent
    in units across versions."""
    try:
        info = yf.Ticker(ticker).info
        y = info.get("dividendYield") or info.get("trailingAnnualDividendYield") or 0.0
        y = float(y)
        # yfinance has changed units across versions (fraction vs percent).
        # A dividend yield above 25% is implausible for a broad index or
        # large-cap stock -- treat that as a signal the value is already
        # in percent form and needs dividing down.
        if y > 0.25:
            y = y / 100.0
        return y
    except Exception as e:
        print(f"  WARNING: could not fetch dividend yield ({e}); using q=0.0")
        return 0.0


def main():
    ticker = "SPY"

    print(f"Fetching data for {ticker}...")
    spot = get_spot(ticker)
    print(f"  Spot price: {spot:.2f}")

    sigma, expiry, days, atm_strike = get_atm_implied_vol(ticker, spot)
    T_years = days / 365.0
    print(f"  ATM implied vol: {sigma*100:.2f}%  (expiry {expiry}, {days} days out, strike {atm_strike})")

    r = get_risk_free_rate(target_years=T_years)
    print(f"  Risk-free rate, interpolated to {T_years:.2f}y: {r*100:.3f}%")

    q = get_dividend_yield(ticker)
    print(f"  Dividend yield: {q*100:.3f}%")

    if abs(days - 365) > 60:
        print(f"  WARNING: closest available expiry is {days} days out, not ~365. "
              f"Consider adjusting T in the pricing model to match, "
              f"or re-run pointing at a ticker with longer-dated options (e.g. SPY, QQQ).")

    result = {
        "ticker": ticker,
        "S0": round(spot, 4),
        "r": round(r, 5),
        "q": round(q, 5),
        "sigma": round(sigma, 5),
        "T_years": round(T_years, 4),
        "expiry_used": expiry,
        "atm_strike_reference": atm_strike,
        "fetched_at": datetime.now().isoformat(),
    }

    print("\n--- Result (already saved, no need to paste it back) ---")
    print(json.dumps(result, indent=2))

    with open("market_params.json", "w") as f:
        json.dump(result, f, indent=2)
    print("\nSaved to market_params.json in the current directory.")
    print("Copy that file next to pricing.py / run_experiment.py, then just run:")
    print("    python run_experiment.py")
    print("It will auto-detect market_params.json and use these real inputs,")
    print("including the dividend yield and term-matched rate, instead of")
    print("the synthetic placeholder defaults.")


if __name__ == "__main__":
    main()


Fetching data for SPY...
  Spot price: 761.69
  ATM implied vol: 23.99%  (expiry 2027-09-17, 361 days out, strike 760.0)
  Risk-free rate, interpolated to 0.99y: 4.115%
  Dividend yield: 0.980%

--- Result (already saved, no need to paste it back) ---
{
  "ticker": "SPY",
  "S0": 761.69,
  "r": 0.04115,
  "q": 0.0098,
  "sigma": 0.23991,
  "T_years": 0.989,
  "expiry_used": "2027-09-17",
  "atm_strike_reference": 760.0,
  "fetched_at": "2026-09-20T21:14:54.311980"
}

Saved to market_params.json in the current directory.
Copy that file next to pricing.py / run_experiment.py, then just run:
    python run_experiment.py
It will auto-detect market_params.json and use these real inputs,
including the dividend yield and term-matched rate, instead of
the synthetic placeholder defaults.


In [2]:
"""
Exotic option pricing: continuously-monitored up-and-out barrier call
(priced via a finite grid, with a bias correction for that discretization),
and a discretely-monitored arithmetic-average Asian call (which is exactly
discrete by contract design -- no correction needed there), under
Black-Scholes/GBM.

Variance reduction techniques implemented:
  - Antithetic variates
  - Control variates (regression-estimated beta, not assumed beta=1)
  - Barrier discretization bias correction (Broadie-Glasserman-Kou
    continuity correction)

All comparisons are done at a FIXED total random-number budget, so
"plain MC" with N paths is compared against antithetic MC using N/2
independent draws (each expanded into a +/- pair), not N vs N.
"""

import numpy as np
from scipy.stats import norm

RNG_SEED = 12345
BGK_ETA = 0.5825972289  # -zeta(1/2)/sqrt(2*pi), Broadie-Glasserman-Kou constant


# ----------------------------------------------------------------------
# Closed-form benchmarks
# ----------------------------------------------------------------------

def bs_call_price(S0, K, r, sigma, T, q=0.0):
    """Black-Scholes European call price with continuous dividend yield q.
    Used as (a) a sanity check and (b) the analytic control-variate mean
    for the barrier option. q=0 recovers the standard non-dividend formula."""
    d1 = (np.log(S0 / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S0 * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def geometric_asian_call_price(S0, K, r, sigma, T, n_steps, q=0.0):
    """Kemna-Vorst (1990) closed form for a discretely-monitored
    geometric-average Asian call, with continuous dividend yield q. This
    is the control variate for the arithmetic-average Asian option below.

    The geometric average of n lognormal observations is itself
    lognormal, so this reduces to a Black-Scholes-style formula with
    adjusted volatility and drift.
    """
    n = n_steps
    sigma_hat = sigma * np.sqrt((n + 1) * (2 * n + 1) / (6 * n ** 2))
    rho = 0.5 * (r - q - 0.5 * sigma ** 2) * (n + 1) / n + 0.5 * sigma_hat ** 2
    d1 = (np.log(S0 / K) + (rho + 0.5 * sigma_hat ** 2) * T) / (sigma_hat * np.sqrt(T))
    d2 = d1 - sigma_hat * np.sqrt(T)
    return np.exp(-r * T) * (S0 * np.exp(rho * T) * norm.cdf(d1) - K * norm.cdf(d2))


# ----------------------------------------------------------------------
# Path simulation
# ----------------------------------------------------------------------

def simulate_gbm_paths(S0, r, sigma, T, n_steps, n_draws, antithetic, rng, q=0.0):
    """Simulate GBM paths using the exact lognormal transition (no
    Euler discretization error in the marginal distribution). Drift is
    risk-neutral: (r - q - 0.5*sigma^2), so an underlying that pays a
    continuous dividend yield q grows more slowly under the pricing
    measure -- q=0 recovers the no-dividend case.

    If antithetic=True, n_draws independent Brownian increment sets are
    each expanded into a (+Z, -Z) pair, returning 2*n_draws paths total.
    This keeps the random-number budget comparable to plain MC with
    2*n_draws paths.
    """
    dt = T / n_steps
    Z = rng.standard_normal((n_draws, n_steps))

    if antithetic:
        Z = np.concatenate([Z, -Z], axis=0)

    increments = (r - q - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * Z
    log_paths = np.cumsum(increments, axis=1)
    S = S0 * np.exp(log_paths)
    S = np.concatenate([np.full((S.shape[0], 1), S0), S], axis=1)
    return S  # shape (n_paths, n_steps + 1)


# ----------------------------------------------------------------------
# Barrier option: discretely-monitored up-and-out call
# ----------------------------------------------------------------------

def _corrected_barrier(B, sigma, dt, is_up):
    """Broadie-Glasserman-Kou continuity correction.

    This corrects for simulating a CONTINUOUSLY monitored barrier option
    on a finite time grid. Checking the barrier only at grid points misses
    within-step excursions, so knock-out probability is underestimated and
    the naive discretized price is biased HIGH relative to the true
    continuous-monitoring price.

    Fix: move the effective barrier TOWARD the spot price (easier to
    trigger in the coarse simulation), which compensates for the missed
    crossings. Up-barriers shift down, down-barriers shift up -- in both
    cases, toward S0.
    """
    sign = -1.0 if is_up else 1.0
    return B * np.exp(sign * BGK_ETA * sigma * np.sqrt(dt))


def barrier_payoff(paths, K, B, r, T, sigma, n_steps, continuity_correction):
    """Discounted payoff of an up-and-out call for each path."""
    dt = T / n_steps
    B_eff = _corrected_barrier(B, sigma, dt, is_up=True) if continuity_correction else B
    knocked_out = np.any(paths >= B_eff, axis=1)
    ST = paths[:, -1]
    payoff = np.maximum(ST - K, 0.0)
    payoff[knocked_out] = 0.0
    return np.exp(-r * T) * payoff


def price_barrier_mc(S0, K, B, r, sigma, T, n_steps, n_draws, method, rng,
                      continuity_correction=True, q=0.0):
    """
    method in {"plain", "antithetic", "control", "antithetic_control"}
    Returns (price, std_error, variance, n_paths_used).
    """
    antithetic = method in ("antithetic", "antithetic_control")
    use_cv = method in ("control", "antithetic_control")

    paths = simulate_gbm_paths(S0, r, sigma, T, n_steps, n_draws, antithetic, rng, q=q)
    Y = barrier_payoff(paths, K, B, r, T, sigma, n_steps, continuity_correction)

    if not use_cv:
        price = Y.mean()
        var = Y.var(ddof=1)
        se = np.sqrt(var / len(Y))
        return price, se, var, len(Y)

    # Control variate: discounted vanilla call payoff on the SAME paths.
    ST = paths[:, -1]
    X = np.exp(-r * T) * np.maximum(ST - K, 0.0)
    EX = bs_call_price(S0, K, r, sigma, T, q=q)

    cov = np.cov(Y, X, ddof=1)[0, 1]
    varX = X.var(ddof=1)
    beta = cov / varX if varX > 0 else 0.0

    Y_cv = Y - beta * (X - EX)
    price = Y_cv.mean()
    var = Y_cv.var(ddof=1)
    se = np.sqrt(var / len(Y_cv))
    return price, se, var, len(Y_cv)


# ----------------------------------------------------------------------
# Asian option: discretely-monitored arithmetic-average call
# ----------------------------------------------------------------------

def asian_payoff(paths, K, r, T, arithmetic=True):
    obs = paths[:, 1:]  # exclude S0 from the averaging window
    avg = obs.mean(axis=1) if arithmetic else np.exp(np.log(obs).mean(axis=1))
    payoff = np.maximum(avg - K, 0.0)
    return np.exp(-r * T) * payoff


def price_asian_mc(S0, K, r, sigma, T, n_steps, n_draws, method, rng, q=0.0):
    """
    method in {"plain", "antithetic", "control", "antithetic_control"}
    """
    antithetic = method in ("antithetic", "antithetic_control")
    use_cv = method in ("control", "antithetic_control")

    paths = simulate_gbm_paths(S0, r, sigma, T, n_steps, n_draws, antithetic, rng, q=q)
    Y = asian_payoff(paths, K, r, T, arithmetic=True)

    if not use_cv:
        price = Y.mean()
        var = Y.var(ddof=1)
        se = np.sqrt(var / len(Y))
        return price, se, var, len(Y)

    X = asian_payoff(paths, K, r, T, arithmetic=False)
    EX = geometric_asian_call_price(S0, K, r, sigma, T, n_steps, q=q)

    cov = np.cov(Y, X, ddof=1)[0, 1]
    varX = X.var(ddof=1)
    beta = cov / varX if varX > 0 else 0.0

    Y_cv = Y - beta * (X - EX)
    price = Y_cv.mean()
    var = Y_cv.var(ddof=1)
    se = np.sqrt(var / len(Y_cv))
    return price, se, var, len(Y_cv)


In [3]:
"""
Before trusting any variance-reduction numbers, verify the simulator
against known closed-form limits. If these fail, the headline results
are meaningless.
"""
import numpy as np

S0, K, r, sigma, T, n_steps = 100.0, 100.0, 0.03, 0.25, 1.0, 52

print("=== Sanity check 1: barrier -> infinity should equal vanilla BS call ===")
rng = np.random.default_rng(RNG_SEED)
bs_price = bs_call_price(S0, K, r, sigma, T)
price, se, _, _ = price_barrier_mc(S0, K, B=1e6, r=r, sigma=sigma, T=T,
                                    n_steps=n_steps, n_draws=100_000,
                                    method="plain", rng=rng,
                                    continuity_correction=False)
print(f"  BS closed form:      {bs_price:.4f}")
print(f"  MC (B=inf):           {price:.4f} +/- {1.96*se:.4f}")
assert abs(price - bs_price) < 3 * se, "FAIL: barrier limit does not match BS price"
print("  PASS\n")

print("=== Sanity check 2: barrier below spot should be worthless ===")
rng = np.random.default_rng(RNG_SEED)
price, se, _, _ = price_barrier_mc(S0, K, B=95.0, r=r, sigma=sigma, T=T,
                                    n_steps=n_steps, n_draws=50_000,
                                    method="plain", rng=rng)
print(f"  MC price (B < S0):    {price:.6f} +/- {1.96*se:.6f}")
assert price < 1e-6, "FAIL: sub-spot barrier should knock out on step 1"
print("  PASS\n")

print("=== Sanity check 3: geometric Asian closed form vs its own MC ===")
rng = np.random.default_rng(RNG_SEED)
geo_price = geometric_asian_call_price(S0, K, r, sigma, T, n_steps)
# reuse asian_payoff machinery directly via price_asian_mc's control path
paths = simulate_gbm_paths(S0, r, sigma, T, n_steps, 200_000, False, rng)
geo_mc = asian_payoff(paths, K, r, T, arithmetic=False)
print(f"  Kemna-Vorst closed form: {geo_price:.4f}")
print(f"  MC estimate:             {geo_mc.mean():.4f} +/- {1.96*geo_mc.std(ddof=1)/np.sqrt(len(geo_mc)):.4f}")
assert abs(geo_mc.mean() - geo_price) < 3 * geo_mc.std(ddof=1) / np.sqrt(len(geo_mc)), \
    "FAIL: geometric Asian MC does not match Kemna-Vorst"
print("  PASS\n")

print("=== Sanity check 4: continuity correction increases knock-out prob (lowers price) ===")
rng1 = np.random.default_rng(RNG_SEED)
rng2 = np.random.default_rng(RNG_SEED)
price_corrected, _, _, _ = price_barrier_mc(S0, K, B=115.0, r=r, sigma=sigma, T=T,
                                             n_steps=n_steps, n_draws=200_000,
                                             method="plain", rng=rng1,
                                             continuity_correction=True)
price_uncorrected, _, _, _ = price_barrier_mc(S0, K, B=115.0, r=r, sigma=sigma, T=T,
                                               n_steps=n_steps, n_draws=200_000,
                                               method="plain", rng=rng2,
                                               continuity_correction=False)
print(f"  Corrected price:   {price_corrected:.4f}")
print(f"  Uncorrected price: {price_uncorrected:.4f}")
assert price_corrected < price_uncorrected, "FAIL: correction should raise knock-out probability and lower price"
print("  PASS (uncorrected grid simulation is biased HIGH -- it misses within-step crossings,\n"
      "        so it understates knock-out probability relative to true continuous monitoring)\n")

print("All sanity checks passed.\n")

print("=== Sanity check 5: dividend yield reduces call value vs q=0, as it should ===")
q_test = 0.02
price_no_div = bs_call_price(S0, K, r, sigma, T, q=0.0)
price_with_div = bs_call_price(S0, K, r, sigma, T, q=q_test)
print(f"  BS call, q=0:     {price_no_div:.4f}")
print(f"  BS call, q={q_test}:  {price_with_div:.4f}")
assert price_with_div < price_no_div, "FAIL: dividends should reduce call value"

rng = np.random.default_rng(RNG_SEED)
mc_price, mc_se, _, _ = price_barrier_mc(S0, K, B=1e6, r=r, sigma=sigma, T=T,
                                          n_steps=n_steps, n_draws=100_000,
                                          method="plain", rng=rng,
                                          continuity_correction=False, q=q_test)
print(f"  MC (B=inf, q={q_test}): {mc_price:.4f} +/- {1.96*mc_se:.4f}  (should match BS call w/ dividends)")
assert abs(mc_price - price_with_div) < 3 * mc_se, "FAIL: dividend-adjusted MC does not match dividend-adjusted BS"
print("  PASS\n")

print("All sanity checks passed.")


=== Sanity check 1: barrier -> infinity should equal vanilla BS call ===
  BS closed form:      11.3485
  MC (B=inf):           11.3495 +/- 0.1114
  PASS

=== Sanity check 2: barrier below spot should be worthless ===
  MC price (B < S0):    0.000000 +/- 0.000000
  PASS

=== Sanity check 3: geometric Asian closed form vs its own MC ===
  Kemna-Vorst closed form: 6.1974
  MC estimate:             6.1918 +/- 0.0414
  PASS

=== Sanity check 4: continuity correction increases knock-out prob (lowers price) ===
  Corrected price:   0.2599
  Uncorrected price: 0.4270
  PASS (uncorrected grid simulation is biased HIGH -- it misses within-step crossings,
        so it understates knock-out probability relative to true continuous monitoring)

All sanity checks passed.

=== Sanity check 5: dividend yield reduces call value vs q=0, as it should ===
  BS call, q=0:     11.3485
  BS call, q=0.02:  10.1975
  MC (B=inf, q=0.02): 10.2012 +/- 0.1054  (should match BS call w/ dividends)
  PASS

All sanit

In [4]:
"""
Fair-budget comparison of plain MC vs antithetic vs control variate vs
both combined, for the barrier and Asian options.

"Fair budget" = every method uses the SAME number of underlying random
draws (n_draws_total independent standard normal path-increments), not
the same number of *paths*. Antithetic variates get n_draws_total/2
independent draws, each expanded into a +/- pair -> n_draws_total paths.
Plain MC gets n_draws_total independent draws -> n_draws_total paths.
This is the only fair comparison; comparing N antithetic paths to N
plain paths overstates the antithetic benefit by roughly 2x because it
silently uses twice the random numbers.

Market inputs: if market_params.json exists (written by
fetch_market_data.py after you run it locally), S0/r/sigma/T are loaded
from there instead of the synthetic placeholder defaults below, and the
run is labeled with the ticker/date it came from. Barrier level is set
as a fixed multiple of spot (not a fixed dollar level) so it's a
sensible barrier regardless of what ticker/spot you calibrate to.
"""
import os
import json
import time
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# Synthetic defaults, used only if market_params.json is not found.
# ---------------------------------------------------------------------
DEFAULT_PARAMS = dict(S0=100.0, K=100.0, r=0.03, q=0.0, sigma=0.25, T=1.0, n_steps=52)
BARRIER_MULTIPLIER = 1.15   # barrier set 15% above spot, up-and-out
K_MULTIPLIER = 1.0          # strike set at-the-money, i.e. K = S0

PARAMS_FILE = "market_params.json"
N_TOTAL_DRAWS = 200_000  # total random path-increment sets, fixed across methods
N_REPS = 20               # independent repetitions, to measure variance of the estimator itself


def load_params():
    """Use real calibrated inputs if fetch_market_data.py has been run
    locally and produced market_params.json; otherwise fall back to the
    synthetic defaults and say so explicitly, so nobody mistakes one
    for the other."""
    if os.path.exists(PARAMS_FILE):
        with open(PARAMS_FILE) as f:
            data = json.load(f)
        S0 = data["S0"]
        r = data["r"]
        q = data.get("q", 0.0)  # older market_params.json files may predate the q field
        sigma = data["sigma"]
        T = data["T_years"]
        source_label = f"{data['ticker']} spot as of {data['fetched_at'][:10]}, expiry {data['expiry_used']}"
        print(f"Loaded CALIBRATED market params from {PARAMS_FILE}: {source_label}")
        return dict(S0=S0, K=S0 * K_MULTIPLIER, r=r, q=q, sigma=sigma, T=T, n_steps=52), source_label
    else:
        print(f"{PARAMS_FILE} not found -- using SYNTHETIC placeholder params. "
              f"Run fetch_market_data.py locally and drop market_params.json "
              f"next to this script to use real calibrated inputs instead.")
        p = dict(DEFAULT_PARAMS)
        return p, "synthetic placeholder values"


def run_method(pricer_fn, method, n_reps, **kwargs):
    prices, ses, variances, times = [], [], [], []
    for rep in range(n_reps):
        rng = np.random.default_rng(RNG_SEED + rep)
        n_draws = N_TOTAL_DRAWS // 2 if method.startswith("antithetic") else N_TOTAL_DRAWS
        t0 = time.perf_counter()
        price, se, var, n_paths = pricer_fn(n_draws=n_draws, method=method, rng=rng, **kwargs)
        elapsed = time.perf_counter() - t0
        prices.append(price)
        ses.append(se)
        variances.append(var)
        times.append(elapsed)
    return np.array(prices), np.array(ses), np.array(variances), np.array(times)


def summarize(name, prices, ses, variances, times, baseline_var=None, baseline_time=None):
    mean_price = prices.mean()
    mean_se = ses.mean()
    mean_var = variances.mean()
    mean_time = times.mean()
    ci_halfwidth = 1.96 * mean_se

    row = {
        "method": name,
        "price": round(mean_price, 4),
        "mc_std_error": round(mean_se, 5),
        "95%_CI_halfwidth": round(ci_halfwidth, 5),
        "per_path_variance": round(mean_var, 5),
        "avg_time_sec": round(mean_time, 4),
    }
    if baseline_var is not None:
        row["variance_reduction_%"] = round(100 * (1 - mean_var / baseline_var), 2)
    if baseline_time is not None and baseline_var is not None:
        eff = (baseline_var / mean_var) / (mean_time / baseline_time)
        row["efficiency_ratio_(var_speedup/time_cost)"] = round(eff, 3)
    return row


def run_suite(label, pricer_fn, **kwargs):
    print(f"\n{'='*70}\n{label}\n{'='*70}")
    methods = ["plain", "antithetic", "control", "antithetic_control"]
    results = {}
    for m in methods:
        results[m] = run_method(pricer_fn, m, N_REPS, **kwargs)

    baseline_var = results["plain"][2].mean()
    baseline_time = results["plain"][3].mean()

    rows = []
    for m in methods:
        prices, ses, variances, times = results[m]
        rows.append(summarize(m, prices, ses, variances, times, baseline_var, baseline_time))

    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return df


if __name__ == "__main__":
    params, source_label = load_params()
    S0, K, r, q, sigma, T, n_steps = (
        params["S0"], params["K"], params["r"], params["q"],
        params["sigma"], params["T"], params["n_steps"]
    )
    B = S0 * BARRIER_MULTIPLIER

    barrier_df = run_suite(
        f"UP-AND-OUT BARRIER CALL  (S0={S0:.2f}, K={K:.2f}, B={B:.2f}, r={r*100:.2f}%, "
        f"q={q*100:.2f}%, sigma={sigma*100:.2f}%, T={T:.2f}y, weekly grid)  [{source_label}]",
        price_barrier_mc,
        S0=S0, K=K, B=B, r=r, sigma=sigma, T=T, n_steps=n_steps,
        continuity_correction=True, q=q,
    )

    asian_df = run_suite(
        f"ARITHMETIC-AVERAGE ASIAN CALL  (S0={S0:.2f}, K={K:.2f}, r={r*100:.2f}%, "
        f"q={q*100:.2f}%, sigma={sigma*100:.2f}%, T={T:.2f}y, weekly grid)  [{source_label}]",
        price_asian_mc,
        S0=S0, K=K, r=r, sigma=sigma, T=T, n_steps=n_steps, q=q,
    )

    barrier_df.to_csv("barrier_results.csv", index=False)
    asian_df.to_csv("asian_results.csv", index=False)
    print("\nSaved barrier_results.csv and asian_results.csv")



Loaded CALIBRATED market params from market_params.json: SPY spot as of 2026-09-20, expiry 2027-09-17

UP-AND-OUT BARRIER CALL  (S0=761.69, K=761.69, B=875.94, r=4.12%, q=0.98%, sigma=23.99%, T=0.99y, weekly grid)  [SPY spot as of 2026-09-20, expiry 2027-09-17]
            method  price  mc_std_error  95%_CI_halfwidth  per_path_variance  avg_time_sec  variance_reduction_%  efficiency_ratio_(var_speedup/time_cost)
             plain 2.1850       0.02342           0.04590          109.68769        0.3051                  0.00                                     1.000
        antithetic 2.1840       0.02340           0.04587          109.52143        0.2776                  0.15                                     1.101
           control 2.1853       0.02339           0.04584          109.42226        0.3081                  0.24                                     0.993
antithetic_control 2.1841       0.02337           0.04581          109.25610        0.2810                  0.39      

In [5]:
"""
Convergence study: how does estimator standard error shrink as N grows,
for plain MC vs control-variate MC, on both option types?

Theory says standard error ~ C / sqrt(N) for any consistent MC estimator
-- the exponent (-0.5 slope on a log-log plot) doesn't change with
variance reduction. What changes is the constant C in front of it: a
good control variate lowers C (shifts the whole line down in log-log
space) without changing the slope. This script checks that empirically
holds, and produces the plot that makes it visible at a glance -- a
number in a table doesn't show this shape as clearly as the plot does.

Uses market_params.json if present (see fetch_market_data.py),
otherwise the same synthetic defaults as run_experiment.py.
"""
import os
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DEFAULT_PARAMS = dict(S0=100.0, K=100.0, r=0.03, q=0.0, sigma=0.25, T=1.0, n_steps=52)
BARRIER_MULTIPLIER = 1.15
PARAMS_FILE = "market_params.json"

N_VALUES = [1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000, 200_000]
N_REPS = 10  # repetitions per N, to average out noise in the SE estimate itself


def load_params():
    if os.path.exists(PARAMS_FILE):
        with open(PARAMS_FILE) as f:
            data = json.load(f)
        S0, r, sigma, T = data["S0"], data["r"], data["sigma"], data["T_years"]
        q = data.get("q", 0.0)
        label = f"{data['ticker']}, {data['fetched_at'][:10]}"
        return dict(S0=S0, K=S0, r=r, q=q, sigma=sigma, T=T, n_steps=52), label
    return dict(DEFAULT_PARAMS, K=DEFAULT_PARAMS["S0"]), "synthetic placeholder values"


def se_vs_n(pricer_fn, method, n_values, n_reps, **kwargs):
    ses = []
    for n in n_values:
        rep_ses = []
        for rep in range(n_reps):
            rng = np.random.default_rng(RNG_SEED + rep)
            n_draws = n // 2 if method.startswith("antithetic") else n
            _, se, _, _ = pricer_fn(n_draws=n_draws, method=method, rng=rng, **kwargs)
            rep_ses.append(se)
        ses.append(np.mean(rep_ses))
    return np.array(ses)


def fit_loglog_slope(n_values, ses):
    log_n = np.log(n_values)
    log_se = np.log(ses)
    slope, intercept = np.polyfit(log_n, log_se, 1)
    return slope, intercept


if __name__ == "__main__":
    params, source_label = load_params()
    B = params["S0"] * BARRIER_MULTIPLIER

    print(f"Using params from: {source_label}")
    print(f"Running convergence study across N = {N_VALUES} ...")

    barrier_plain = se_vs_n(price_barrier_mc, "plain", N_VALUES, N_REPS,
                             S0=params["S0"], K=params["K"], B=B, r=params["r"],
                             sigma=params["sigma"], T=params["T"], n_steps=params["n_steps"],
                             q=params["q"])
    barrier_cv = se_vs_n(price_barrier_mc, "control", N_VALUES, N_REPS,
                          S0=params["S0"], K=params["K"], B=B, r=params["r"],
                          sigma=params["sigma"], T=params["T"], n_steps=params["n_steps"],
                          q=params["q"])
    asian_plain = se_vs_n(price_asian_mc, "plain", N_VALUES, N_REPS,
                           S0=params["S0"], K=params["K"], r=params["r"],
                           sigma=params["sigma"], T=params["T"], n_steps=params["n_steps"],
                           q=params["q"])
    asian_cv = se_vs_n(price_asian_mc, "control", N_VALUES, N_REPS,
                        S0=params["S0"], K=params["K"], r=params["r"],
                        sigma=params["sigma"], T=params["T"], n_steps=params["n_steps"],
                        q=params["q"])

    print("\nFitted log-log slopes (theory predicts -0.5 for all four lines):")
    for name, ses in [("Barrier, plain", barrier_plain), ("Barrier, control", barrier_cv),
                       ("Asian, plain", asian_plain), ("Asian, control", asian_cv)]:
        slope, _ = fit_loglog_slope(N_VALUES, ses)
        print(f"  {name:20s}: slope = {slope:.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].loglog(N_VALUES, barrier_plain, 'o-', label='Plain MC')
    axes[0].loglog(N_VALUES, barrier_cv, 's-', label='Control variate (vanilla call)')
    axes[0].set_xlabel('N (paths)')
    axes[0].set_ylabel('Standard error')
    axes[0].set_title('Barrier option: SE vs N\n(lines nearly overlap -- control variate barely helps)')
    axes[0].legend()
    axes[0].grid(True, which='both', alpha=0.3)

    axes[1].loglog(N_VALUES, asian_plain, 'o-', label='Plain MC')
    axes[1].loglog(N_VALUES, asian_cv, 's-', label='Control variate (geometric Asian)')
    axes[1].set_xlabel('N (paths)')
    axes[1].set_ylabel('Standard error')
    axes[1].set_title('Asian option: SE vs N\n(control variate line sits ~30x lower, same slope)')
    axes[1].legend()
    axes[1].grid(True, which='both', alpha=0.3)

    plt.tight_layout()
    plt.savefig('convergence_study.png', dpi=150)
    print("\nSaved convergence_study.png")


Using params from: SPY, 2026-09-20
Running convergence study across N = [1000, 2000, 5000, 10000, 20000, 50000, 100000, 200000] ...

Fitted log-log slopes (theory predicts -0.5 for all four lines):
  Barrier, plain      : slope = -0.498
  Barrier, control    : slope = -0.498
  Asian, plain        : slope = -0.499
  Asian, control      : slope = -0.494

Saved convergence_study.png


In [6]:
"""
Domain-specific variance reduction for the up-and-out barrier call,
addressing why plain antithetic/control-variate tricks failed (see
REPORT.md): the knock-out indicator is a discontinuous function of the
path, which breaks both techniques.

Three techniques, in order of what actually gets built:

1. Brownian bridge conditional expectation (Rao-Blackwellization).
   Instead of checking whether simulated GRID POINTS breach the barrier,
   compute the exact analytical probability that a continuous Brownian
   bridge between each pair of consecutive grid points crosses the
   barrier, and multiply survival probabilities across all intervals.
   This does two things at once: it removes the discretization bias
   (no more approximate BGK correction needed -- this IS the continuous
   monitoring probability, not an approximation of it), and it reduces
   variance by the law of total variance: Var(E[X|skeleton]) <= Var(X).
   This replaces the 0/1 knock-out indicator with a smooth number in
   [0, 1], which is why it helps -- discontinuities are what killed
   antithetic variates and the vanilla-call control variate in the
   original approach.

2. Reiner-Rubinstein closed-form continuous-barrier price. Used ONLY as
   an independent validation benchmark for the Brownian bridge
   estimator above -- NOT plugged in as a control-variate mean, because
   the naive discrete payoff and the continuous closed-form price
   differ by a bias term, not just noise, so using it that way would be
   silently wrong. This formula is reconstructed from memory and
   validated against high-N simulation below before being trusted for
   anything; if it doesn't match, the mismatch is reported rather than
   hidden.

3. Importance sampling via drift shift. NOTE: my first-pass reasoning
   here was wrong and worth recording rather than quietly fixing. The
   naive argument is "the barrier gets hit often, so surviving paths are
   the rare event -- shift drift away from the barrier to sample more of
   them." Empirically, that direction makes variance WORSE, not better,
   because it only accounts for one constraint (avoid knock-out) while
   ignoring the other (still needs to end above K). Since K = S0 here
   (at-the-money), shifting drift down to avoid the barrier also drags
   terminal price below the strike more often, killing the payoff
   through the exact channel the shift was meant to protect. This
   option's value lives in a narrow band of paths that rise enough to be
   ITM but not so much they hit the barrier -- a downward shift doesn't
   preserve that band, it destroys it. Empirically, a SMALL shift TOWARD
   the barrier gives a modest (~8-9%) variance reduction; large shifts in
   either direction make things worse. The calibrate_drift_shift()
   function below finds a reasonable value by pilot search rather than
   assuming a sign from theory alone -- a two-constraint rare-event
   problem (survive AND finish ITM) doesn't reduce to the single-event
   textbook case cleanly.
"""

import numpy as np
from scipy.stats import norm



# ----------------------------------------------------------------------
# 1. Brownian bridge conditional expectation
# ----------------------------------------------------------------------

def bridge_survival_probability(paths, B, sigma, dt):
    """
    For each path, the probability that a Brownian bridge between each
    pair of consecutive observed points stays BELOW the upper barrier B
    throughout the interval, multiplied across all intervals.

    Standard result (see e.g. Glasserman, "Monte Carlo Methods in
    Financial Engineering", Sec 6.4): for a GBM path observed at S_i and
    S_{i+1} over an interval of length dt, conditional on those two
    endpoints, the probability the continuous path crosses an upper
    barrier B (with both S_i < B and S_{i+1} < B) is:

        p_cross = exp( -2 * ln(B/S_i) * ln(B/S_{i+1}) / (sigma^2 * dt) )

    If either endpoint is already >= B, the path has certainly crossed
    (crossing probability 1, survival 0).
    """
    S_i = paths[:, :-1]
    S_next = paths[:, 1:]

    already_breached = (S_i >= B) | (S_next >= B)

    # only compute the log-ratio where both endpoints are safely below B,
    # to avoid log(<=0) on entries that will be overridden anyway
    safe = ~already_breached
    log_ratio_i = np.where(safe, np.log(np.clip(B, 1e-300, None) / np.clip(S_i, 1e-300, None)), 0.0)
    log_ratio_next = np.where(safe, np.log(np.clip(B, 1e-300, None) / np.clip(S_next, 1e-300, None)), 0.0)

    p_cross = np.exp(-2.0 * log_ratio_i * log_ratio_next / (sigma ** 2 * dt))
    p_cross = np.where(already_breached, 1.0, p_cross)
    p_survive_interval = 1.0 - p_cross

    # product of per-interval survival probabilities = probability the
    # WHOLE path never crosses, conditional on the observed skeleton
    return np.prod(p_survive_interval, axis=1)


def barrier_payoff_bridge(paths, K, B, r, T, sigma, n_steps):
    """Rao-Blackwellized discounted payoff: discounted vanilla payoff at
    S_T, weighted by the exact conditional probability of never crossing
    the barrier given the simulated skeleton. Replaces the discontinuous
    0/1 knock-out indicator with a smooth number in [0, 1] -- this is
    what fixes the variance-reduction failure from the naive approach."""
    dt = T / n_steps
    survival = bridge_survival_probability(paths, B, sigma, dt)
    ST = paths[:, -1]
    vanilla_payoff = np.maximum(ST - K, 0.0)
    return np.exp(-r * T) * vanilla_payoff * survival


def price_barrier_bridge_mc(S0, K, B, r, sigma, T, n_steps, n_draws, method, rng, q=0.0):
    """
    Barrier pricer using the Brownian-bridge conditional payoff instead
    of the naive discrete indicator. method in
    {"plain", "antithetic", "control", "antithetic_control"} -- same
    method names as the original pricer, but the underlying payoff
    computation is fundamentally different (smooth, not discontinuous).
    """
    antithetic = method in ("antithetic", "antithetic_control")
    use_cv = method in ("control", "antithetic_control")

    paths = simulate_gbm_paths(S0, r, sigma, T, n_steps, n_draws, antithetic, rng, q=q)
    Y = barrier_payoff_bridge(paths, K, B, r, T, sigma, n_steps)

    if not use_cv:
        price = Y.mean()
        var = Y.var(ddof=1)
        se = np.sqrt(var / len(Y))
        return price, se, var, len(Y)

    # Control variate: discounted vanilla call payoff, same as the
    # original approach. Weaker justification here than technique (1)
    # itself, but included for a like-for-like comparison.
    ST = paths[:, -1]
    X = np.exp(-r * T) * np.maximum(ST - K, 0.0)
    EX = bs_call_price(S0, K, r, sigma, T, q=q)

    cov = np.cov(Y, X, ddof=1)[0, 1]
    varX = X.var(ddof=1)
    beta = cov / varX if varX > 0 else 0.0

    Y_cv = Y - beta * (X - EX)
    price = Y_cv.mean()
    var = Y_cv.var(ddof=1)
    se = np.sqrt(var / len(Y_cv))
    return price, se, var, len(Y_cv)


# ----------------------------------------------------------------------
# 2. Closed-form continuous-barrier price (Reiner-Rubinstein / Merton)
#    -- validation benchmark ONLY, not a control-variate mean.
# ----------------------------------------------------------------------

def up_and_out_call_closed_form(S0, K, B, r, sigma, T, q=0.0):
    """
    Closed-form price of a CONTINUOUSLY monitored up-and-out European
    call, for the case B > K (barrier above strike), no rebate.
    Reiner & Rubinstein (1991) / Merton (1973) formula, as tabulated in
    Haug's "Complete Guide to Option Pricing Formulas".

    This is reconstructed from memory, not copied from a live reference,
    so it is validated against high-N simulation in sanity_checks.py
    before being trusted anywhere else in this project. If validation
    fails, that failure is reported, not hidden.
    """
    if B <= K:
        raise ValueError("This formula assumes B > K (barrier above strike); "
                          "a different formula branch is needed otherwise.")

    b = r - q
    mu = (b - 0.5 * sigma ** 2) / sigma ** 2
    sig_sqrtT = sigma * np.sqrt(T)

    x1 = np.log(S0 / K) / sig_sqrtT + (1 + mu) * sig_sqrtT
    x2 = np.log(S0 / B) / sig_sqrtT + (1 + mu) * sig_sqrtT
    y1 = np.log(B ** 2 / (S0 * K)) / sig_sqrtT + (1 + mu) * sig_sqrtT
    y2 = np.log(B / S0) / sig_sqrtT + (1 + mu) * sig_sqrtT

    term1 = S0 * np.exp(-q * T) * (norm.cdf(x1) - norm.cdf(x2))
    term2 = -K * np.exp(-r * T) * (norm.cdf(x1 - sig_sqrtT) - norm.cdf(x2 - sig_sqrtT))
    term3 = -S0 * np.exp(-q * T) * (B / S0) ** (2 * (mu + 1)) * (norm.cdf(y1) - norm.cdf(y2))
    term4 = K * np.exp(-r * T) * (B / S0) ** (2 * mu) * (norm.cdf(y1 - sig_sqrtT) - norm.cdf(y2 - sig_sqrtT))

    return term1 + term2 + term3 + term4


# ----------------------------------------------------------------------
# 3. Importance sampling via drift shift
# ----------------------------------------------------------------------

def simulate_gbm_paths_tilted(S0, r, sigma, T, n_steps, n_draws, rng, drift_shift, q=0.0):
    """
    Simulate paths under a drift-SHIFTED measure Q, and return both the
    paths and the likelihood ratio L = dP/dQ needed to convert a
    Q-expectation back into the correct (unbiased) P-expectation.

    drift_shift is added to the risk-neutral drift (r - q - 0.5*sigma^2)
    for the PURPOSE OF SAMPLING ONLY -- it changes which paths get drawn,
    not the model's actual dynamics. Positive drift_shift pushes paths
    UP (toward an upper barrier); negative pushes them down (away from
    it). For an up-and-out call where knock-outs are already common,
    drift_shift should be NEGATIVE (push away from the barrier, generate
    more surviving/ITM paths) -- see module docstring.

    Derivation: raw standard normals eps ~ N(0,1) are shifted by a
    constant c = drift_shift*sqrt(dt)/sigma before being used to build
    the path (this adds exactly drift_shift*dt of extra drift per step,
    drift_shift*T total, since summing n_steps identical per-step shifts
    of size sigma*c*sqrt(dt) = drift_shift*dt gives drift_shift*T over
    the whole period). The likelihood ratio for n_steps i.i.d. shocks
    mean-shifted by c is L = exp(-c*sum(eps) - 0.5*n_steps*c^2), which
    reweights a Q-average back to the correct P-expectation.
    """
    dt = T / n_steps
    c = drift_shift * np.sqrt(dt) / sigma

    eps = rng.standard_normal((n_draws, n_steps))
    z_tilted = eps + c

    increments = (r - q - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * z_tilted
    log_paths = np.cumsum(increments, axis=1)
    S = S0 * np.exp(log_paths)
    S = np.concatenate([np.full((S.shape[0], 1), S0), S], axis=1)

    likelihood_ratio = np.exp(-c * eps.sum(axis=1) - 0.5 * n_steps * c ** 2)
    return S, likelihood_ratio


def price_barrier_importance_sampling(S0, K, B, r, sigma, T, n_steps, n_draws, rng,
                                       drift_shift, use_bridge=True, q=0.0):
    """
    Importance-sampled barrier price. use_bridge=True combines this with
    the Brownian-bridge payoff (1) for the strongest available combo;
    use_bridge=False uses the naive discrete indicator, to isolate how
    much importance sampling contributes on its own.

    Returns (price, std_error, variance, n_paths_used).
    """
    paths, L = simulate_gbm_paths_tilted(S0, r, sigma, T, n_steps, n_draws, rng, drift_shift, q=q)

    if use_bridge:
        dt = T / n_steps
        survival = bridge_survival_probability(paths, B, sigma, dt)
        ST = paths[:, -1]
        Y_raw = np.exp(-r * T) * np.maximum(ST - K, 0.0) * survival
    else:
        knocked_out = np.any(paths >= B, axis=1)
        ST = paths[:, -1]
        payoff = np.maximum(ST - K, 0.0)
        payoff[knocked_out] = 0.0
        Y_raw = np.exp(-r * T) * payoff

    Y = Y_raw * L
    price = Y.mean()
    var = Y.var(ddof=1)
    se = np.sqrt(var / len(Y))
    return price, se, var, len(Y)


def calibrate_drift_shift(S0, K, B, r, sigma, T, n_steps, r_state, use_bridge=True,
                           q=0.0, pilot_n=20_000, candidate_shifts=None):
    """
    Pick a drift shift by pilot search rather than assuming a sign from
    theory -- see the module docstring for why the naive "shift away
    from the barrier" argument doesn't hold here. Runs a small pilot at
    each candidate shift (shared random numbers across candidates, for a
    fair comparison) and returns the shift with the lowest observed
    variance.

    r_state: an integer seed (not an np.random.Generator) so each
    candidate can be evaluated on the SAME underlying draws.
    """
    if candidate_shifts is None:
        candidate_shifts = [-0.20, -0.10, -0.05, 0.0, 0.05, 0.10, 0.20]

    best_shift, best_var = 0.0, None
    results = []
    for shift in candidate_shifts:
        rng = np.random.default_rng(r_state)
        if shift == 0.0:
            paths = simulate_gbm_paths(S0, r, sigma, T, n_steps, pilot_n, False, rng, q=q)
            if use_bridge:
                dt = T / n_steps
                survival = bridge_survival_probability(paths, B, sigma, dt)
                ST = paths[:, -1]
                Y = np.exp(-r * T) * np.maximum(ST - K, 0.0) * survival
            else:
                knocked_out = np.any(paths >= B, axis=1)
                ST = paths[:, -1]
                payoff = np.maximum(ST - K, 0.0)
                payoff[knocked_out] = 0.0
                Y = np.exp(-r * T) * payoff
            var = Y.var(ddof=1)
        else:
            _, _, var, _ = price_barrier_importance_sampling(
                S0, K, B, r, sigma, T, n_steps, pilot_n, rng, shift, use_bridge=use_bridge, q=q)
        results.append((shift, var))
        if best_var is None or var < best_var:
            best_shift, best_var = shift, var

    return best_shift, results


In [7]:
"""
Validation for barrier_advanced.py, BEFORE trusting any of it in the
main experiment. The closed-form formula in particular was reconstructed
from memory and needs to earn trust against simulation, not be assumed
correct because it looks like a textbook formula.
"""
import numpy as np

S0, K, r, sigma, T, n_steps, q = 100.0, 100.0, 0.03, 0.25, 1.0, 52, 0.0
B = 115.0

print("=== Check 1: closed-form up-and-out price vs high-N Brownian-bridge MC ===")
rng = np.random.default_rng(999)
price_mc, se_mc, _, _ = price_barrier_bridge_mc(S0, K, B, r, sigma, T, n_steps,
                                                 n_draws=500_000, method="plain", rng=rng, q=q)
price_cf = up_and_out_call_closed_form(S0, K, B, r, sigma, T, q=q)
print(f"  Closed form:          {price_cf:.4f}")
print(f"  Bridge MC (N=500k):   {price_mc:.4f} +/- {1.96*se_mc:.4f}")
diff = abs(price_cf - price_mc)
within_ci = diff < 3 * se_mc
print(f"  Difference: {diff:.4f}  ({'within' if within_ci else 'OUTSIDE'} 3 std errors)")
if not within_ci:
    print("  WARNING: closed-form formula does NOT match simulation. "
          "Treating the closed form as unreliable -- do not use it elsewhere "
          "until this is resolved.")
else:
    print("  PASS: closed-form formula validated against independent simulation.")

print("\n=== Check 2: closed-form up-and-out price vs high-N NAIVE discrete MC ===")
print("    (expect naive discrete to be slightly HIGHER than continuous closed form,")
print("     since discrete monitoring misses some crossings)")
rng = np.random.default_rng(998)
paths = simulate_gbm_paths(S0, r, sigma, T, n_steps, 500_000, False, rng, q=q)
knocked_out = np.any(paths >= B, axis=1)
ST = paths[:, -1]
payoff = np.maximum(ST - K, 0.0)
payoff[knocked_out] = 0.0
naive_price = (np.exp(-r*T) * payoff).mean()
naive_se = (np.exp(-r*T) * payoff).std(ddof=1) / np.sqrt(len(payoff))
print(f"  Naive discrete MC:    {naive_price:.4f} +/- {1.96*naive_se:.4f}")
print(f"  Closed form (cont.):  {price_cf:.4f}")
print(f"  Naive > continuous?   {naive_price > price_cf}  (expected: True)")

print("\n=== Check 3: Brownian bridge survival prob is between 0 and 1, and 1 when far from barrier ===")
rng = np.random.default_rng(1)
paths = simulate_gbm_paths(50.0, r, sigma, T, n_steps, 1000, False, rng, q=q)  # spot far below B=115
surv = bridge_survival_probability(paths, B, sigma, T/n_steps)
print(f"  Spot=50 (far from B=115): min survival={surv.min():.6f}, max={surv.max():.6f}, mean={surv.mean():.6f}")
assert (surv >= 0).all() and (surv <= 1).all(), "FAIL: survival probability out of [0,1] range"
assert surv.mean() > 0.99, "FAIL: paths far from barrier should almost always survive"
print("  PASS")

print("\n=== Check 4: importance sampling stays unbiased regardless of drift shift sign/size ===")
print("    (this is the real invariant to test -- NOT a claimed direction for variance,")
print("     since the naive 'shift away from barrier' argument turned out to be wrong;")
print("     see module docstring and calibrate_drift_shift() for the honest version)")
rng1 = np.random.default_rng(42)
plain_price, plain_se, plain_var, _ = price_barrier_bridge_mc(
    S0, K, B, r, sigma, T, n_steps, n_draws=100_000, method="plain", rng=rng1, q=q)

all_unbiased = True
for shift in [-0.10, 0.05, 0.10, 0.20]:
    rng2 = np.random.default_rng(42)
    is_price, is_se, is_var, _ = price_barrier_importance_sampling(
        S0, K, B, r, sigma, T, n_steps, n_draws=100_000, rng=rng2,
        drift_shift=shift, use_bridge=True, q=q)
    agrees = abs(plain_price - is_price) < 4 * (plain_se + is_se)
    all_unbiased = all_unbiased and agrees
    print(f"  shift={shift:+.2f}: price={is_price:.4f} (plain={plain_price:.4f}), "
          f"var={is_var:.4f} (plain={plain_var:.4f}), unbiased={agrees}")
assert all_unbiased, "FAIL: importance sampling should be unbiased regardless of shift chosen"
print("  PASS: reweighted estimator matches plain bridge MC at every tested shift.")

print("\n=== Check 5: calibrate_drift_shift() finds a shift at least as good as no shift ===")
best_shift, sweep_results = calibrate_drift_shift(
    S0, K, B, r, sigma, T, n_steps, r_state=123, use_bridge=True, q=q, pilot_n=20_000)
print(f"  Sweep results (shift, variance): {[(s, round(v,3)) for s, v in sweep_results]}")
print(f"  Best shift found: {best_shift:+.2f}")
baseline_var = dict(sweep_results)[0.0]
best_var = dict(sweep_results)[best_shift]
print(f"  Best variance {best_var:.4f} <= baseline (shift=0) variance {baseline_var:.4f}: "
      f"{best_var <= baseline_var}")
assert best_var <= baseline_var, "FAIL: calibrated shift should be no worse than no shift"
print("  PASS")

print("\nAll barrier_advanced.py validation checks complete.")


=== Check 1: closed-form up-and-out price vs high-N Brownian-bridge MC ===
  Closed form:          0.2676
  Bridge MC (N=500k):   0.2684 +/- 0.0035
  Difference: 0.0008  (within 3 std errors)
  PASS: closed-form formula validated against independent simulation.

=== Check 2: closed-form up-and-out price vs high-N NAIVE discrete MC ===
    (expect naive discrete to be slightly HIGHER than continuous closed form,
     since discrete monitoring misses some crossings)
  Naive discrete MC:    0.4187 +/- 0.0050
  Closed form (cont.):  0.2676
  Naive > continuous?   True  (expected: True)

=== Check 3: Brownian bridge survival prob is between 0 and 1, and 1 when far from barrier ===
  Spot=50 (far from B=115): min survival=1.000000, max=1.000000, mean=1.000000
  PASS

=== Check 4: importance sampling stays unbiased regardless of drift shift sign/size ===
    (this is the real invariant to test -- NOT a claimed direction for variance,
     since the naive 'shift away from barrier' argument tur

In [8]:
"""
Two further refinements to the barrier variance-reduction work in
barrier_advanced.py:

1. Survival-based control variate. Uses the discounted survival
   probability ITSELF (not multiplied by the payoff) as the control
   variate, with its own closed-form mean -- a discounted "no-touch"
   digital barrier price. This is a genuinely different quantity from
   the target Y, not a repackaging of the same closed-form call price
   used as both target and control (which would trivially collapse
   variance to ~0 by construction, without demonstrating anything about
   the technique -- see module-level warning below). Its actual
   correlation with Y is an empirical question, tested here rather than
   assumed.

2. State-dependent drift tilting. NOT a reproduction of the Glasserman-
   Heidelberger-Shahabuddin asymptotically-optimal large-deviations
   tilt -- deriving that requires solving a variational problem specific
   to this exact payoff, which is out of scope here. Instead: a
   "corridor-pinning" heuristic, where the drift shift at each step is
   recomputed from the CURRENT state (spot, time remaining) to steer the
   expected terminal log-price toward the geometric center of (K, B).
   The likelihood-ratio math for a state-dependent (predictable) drift
   shift is a direct, valid generalization of the constant-shift formula
   in barrier_advanced.py -- as long as the shift at each step is
   computed from information available BEFORE that step's random draw
   (i.e., from S_i and t_i, not from S_{i+1}), each step's local
   Radon-Nikodym factor is still exp(-c_i*eps_i - 0.5*c_i^2), and the
   product across steps is still a valid, unbiased change of measure.
   This is checked directly below (unbiasedness), not assumed, given
   that the constant-shift version of this exact kind of code had two
   real bugs before it was trusted.
"""

import numpy as np
from scipy.stats import norm



# ----------------------------------------------------------------------
# 1. Survival-based control variate
# ----------------------------------------------------------------------

def no_touch_discounted_price(S0, B, r, sigma, T, q=0.0):
    """
    Closed-form price of a discounted "no-touch" claim: pays $1 at T if
    the continuously-monitored path never reaches the upper barrier B,
    else $0. This is the E[X] needed for the survival-based control
    variate -- a genuinely different closed form from the vanilla-call-
    based up_and_out_call_closed_form in barrier_advanced.py (that one
    has strike-dependent terms; this one doesn't, since it's a pure
    survival probability, not a payoff-weighted one).

    Standard reflection-principle result for GBM (see e.g. Shreve,
    "Stochastic Calculus for Finance II", or Haug's barrier binary
    formulas): under drift mu = (r - q - 0.5*sigma^2),

        P(max_t S_t < B) = N(d) - (B/S0)^(2*mu/sigma^2) * N(d')

    where d and d' are the usual barrier reflection terms. Reconstructed
    from memory, so validated against high-N simulation immediately
    below before being trusted as a control-variate mean.
    """
    mu = r - q - 0.5 * sigma ** 2
    sig_sqrtT = sigma * np.sqrt(T)

    d = (np.log(B / S0) - mu * T) / sig_sqrtT
    d_prime = (np.log(B / S0) + mu * T) / sig_sqrtT  # reflection term

    p_no_touch = norm.cdf(d) - (B / S0) ** (2 * mu / sigma ** 2) * norm.cdf(-d_prime)
    return np.exp(-r * T) * p_no_touch


def price_barrier_bridge_survival_cv(S0, K, B, r, sigma, T, n_steps, n_draws, rng, q=0.0):
    """
    Brownian-bridge barrier price with a survival-based control variate:
    X = discounted per-path survival probability (NOT multiplied by the
    vanilla payoff), EX = no_touch_discounted_price(...). Genuinely
    different quantity from Y, correlation tested empirically.
    """
    paths = simulate_gbm_paths(S0, r, sigma, T, n_steps, n_draws, False, rng, q=q)
    dt = T / n_steps
    survival = bridge_survival_probability(paths, B, sigma, dt)
    ST = paths[:, -1]

    Y = np.exp(-r * T) * np.maximum(ST - K, 0.0) * survival
    X = np.exp(-r * T) * survival
    EX = no_touch_discounted_price(S0, B, r, sigma, T, q=q)

    cov = np.cov(Y, X, ddof=1)[0, 1]
    varX = X.var(ddof=1)
    beta = cov / varX if varX > 0 else 0.0
    correlation = cov / np.sqrt(Y.var(ddof=1) * varX) if varX > 0 else 0.0

    Y_cv = Y - beta * (X - EX)
    price = Y_cv.mean()
    var = Y_cv.var(ddof=1)
    se = np.sqrt(var / len(Y_cv))
    return price, se, var, len(Y_cv), correlation


# ----------------------------------------------------------------------
# 2. State-dependent (corridor-pinning) drift tilting
# ----------------------------------------------------------------------

def simulate_gbm_paths_state_dependent_tilt(S0, K, B, r, sigma, T, n_steps, n_draws, rng, q=0.0,
                                             strength=1.0):
    """
    Simulate paths with a drift shift recomputed at EVERY STEP from the
    current state, steering the expected terminal log-price toward the
    geometric center of (K, B), sqrt(K*B). `strength` in [0, 1] scales
    how aggressively the tilt pursues that target (1.0 = fully pin to
    the target given current drift and time remaining; 0.0 = no tilt,
    recovers plain simulation).

    Returns (paths, likelihood_ratio). Likelihood ratio is accumulated
    step by step: since each step's shift c_i is computed from S_i and
    t_i (available BEFORE that step's random draw), the same per-step
    Radon-Nikodym factor from the constant-shift case applies at each
    step, and the product across steps is a valid, unbiased change of
    measure -- this is checked directly in the validation script, not
    assumed.
    """
    dt = T / n_steps
    target_log = 0.5 * (np.log(K) + np.log(B))
    drift = r - q - 0.5 * sigma ** 2

    S = np.full(n_draws, S0)
    log_L = np.zeros(n_draws)
    path_history = [S.copy()]

    for i in range(n_steps):
        tau_remaining = T - i * dt
        # what plain (untilted) drift would deliver by expiry, from here
        implied_terminal_log = np.log(S) + drift * tau_remaining
        # total EXTRA drift (not rate) needed over remaining time to hit target
        total_extra_drift_needed = strength * (target_log - implied_terminal_log)
        # convert to a rate, applied only for this one step's remaining horizon
        drift_shift_rate = np.where(tau_remaining > 1e-12,
                                     total_extra_drift_needed / tau_remaining, 0.0)
        c_i = drift_shift_rate * np.sqrt(dt) / sigma  # per-step shift constant, this step's state

        eps_i = rng.standard_normal(n_draws)
        z_i = eps_i + c_i

        increment = drift * dt + sigma * np.sqrt(dt) * z_i
        S = S * np.exp(increment)
        path_history.append(S.copy())

        log_L += -(c_i * eps_i + 0.5 * c_i ** 2)

    paths = np.stack(path_history, axis=1)
    likelihood_ratio = np.exp(log_L)
    return paths, likelihood_ratio


def price_barrier_state_dependent_is(S0, K, B, r, sigma, T, n_steps, n_draws, rng, q=0.0,
                                      strength=1.0, use_bridge=True):
    """
    Barrier price using the state-dependent (corridor-pinning) tilt.
    Returns (price, std_error, variance, n_paths_used).
    """
    paths, L = simulate_gbm_paths_state_dependent_tilt(
        S0, K, B, r, sigma, T, n_steps, n_draws, rng, q=q, strength=strength)

    if use_bridge:
        dt = T / n_steps
        survival = bridge_survival_probability(paths, B, sigma, dt)
        ST = paths[:, -1]
        Y_raw = np.exp(-r * T) * np.maximum(ST - K, 0.0) * survival
    else:
        knocked_out = np.any(paths >= B, axis=1)
        ST = paths[:, -1]
        payoff = np.maximum(ST - K, 0.0)
        payoff[knocked_out] = 0.0
        Y_raw = np.exp(-r * T) * payoff

    Y = Y_raw * L
    price = Y.mean()
    var = Y.var(ddof=1)
    se = np.sqrt(var / len(Y))
    return price, se, var, len(Y)


def calibrate_tilt_strength(S0, K, B, r, sigma, T, n_steps, r_state, q=0.0,
                             pilot_n=20_000, candidate_strengths=None):
    """Pilot search for the pinning strength, same pattern as
    calibrate_drift_shift in barrier_advanced.py -- no first-principles
    derivation of the optimum, just an honest empirical search."""
    if candidate_strengths is None:
        candidate_strengths = [0.0, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

    best_strength, best_var = 0.0, None
    results = []
    for strength in candidate_strengths:
        rng = np.random.default_rng(r_state)
        _, _, var, _ = price_barrier_state_dependent_is(
            S0, K, B, r, sigma, T, n_steps, pilot_n, rng, q=q, strength=strength)
        results.append((strength, var))
        if best_var is None or var < best_var:
            best_strength, best_var = strength, var

    return best_strength, results


In [10]:
"""
Validation for barrier_state_dependent.py. Given the track record in
this project (two real bugs in the constant-shift IS code before it was
trusted), nothing here gets used in the main comparison until it passes
these checks.
"""
import numpy as np


S0, K, r, sigma, T, n_steps, q = 100.0, 100.0, 0.03, 0.25, 1.0, 52, 0.0
B = 115.0

print("=== Check 1: no-touch closed form vs high-N survival-probability MC ===")
rng = np.random.default_rng(555)
paths = simulate_gbm_paths(S0, r, sigma, T, n_steps, 500_000, False, rng, q=q)
dt = T / n_steps
survival = bridge_survival_probability(paths, B, sigma, dt)
mc_price = (np.exp(-r * T) * survival).mean()
mc_se = (np.exp(-r * T) * survival).std(ddof=1) / np.sqrt(len(survival))
cf_price = no_touch_discounted_price(S0, B, r, sigma, T, q=q)
print(f"  Closed form:        {cf_price:.5f}")
print(f"  Bridge MC (N=500k): {mc_price:.5f} +/- {1.96*mc_se:.5f}")
diff = abs(cf_price - mc_price)
within_ci = diff < 3 * mc_se
print(f"  Difference: {diff:.5f}  ({'within' if within_ci else 'OUTSIDE'} 3 std errors)")
if not within_ci:
    print("  WARNING: no-touch closed form does not match simulation -- NOT reliable, do not use.")
else:
    print("  PASS")

print("\n=== Check 2: survival-based control variate is unbiased, and its actual correlation ===")
rng = np.random.default_rng(1)
price_cv, se_cv, var_cv, n, corr = price_barrier_bridge_survival_cv(
    S0, K, B, r, sigma, T, n_steps, n_draws=100_000, rng=rng, q=q)
cf_call_price = up_and_out_call_closed_form(S0, K, B, r, sigma, T, q=q)
print(f"  Closed-form call price:         {cf_call_price:.4f}")
print(f"  Survival-CV estimate:           {price_cv:.4f} +/- {1.96*se_cv:.4f}")
agrees = abs(price_cv - cf_call_price) < 4 * se_cv
print(f"  Agrees with closed form: {agrees}")
print(f"  Empirical correlation(Y, survival): {corr:.4f}")
print(f"  (this is the number that actually matters -- not assumed, measured)")

print("\n=== Check 3: state-dependent tilt is unbiased, at several 'strength' settings ===")
rng_baseline = np.random.default_rng(2)

baseline_price, baseline_se, baseline_var, _ = price_barrier_bridge_mc(
    S0, K, B, r, sigma, T, n_steps, n_draws=200_000, method="plain", rng=rng_baseline, q=q)
print(f"  Baseline (plain bridge, N=200k): price={baseline_price:.4f} +/- {1.96*baseline_se:.4f}")

all_unbiased = True
for strength in [0.0, 0.3, 0.6, 1.0]:
    rng = np.random.default_rng(3)
    price, se, var, _ = price_barrier_state_dependent_is(
        S0, K, B, r, sigma, T, n_steps, n_draws=100_000, rng=rng, q=q, strength=strength)
    agrees = abs(price - baseline_price) < 4 * (se + baseline_se)
    all_unbiased = all_unbiased and agrees
    print(f"  strength={strength:.1f}: price={price:.4f} +/- {1.96*se:.4f}  "
          f"var={var:.4f}  unbiased={agrees}")
assert all_unbiased, "FAIL: state-dependent tilt should be unbiased at every strength"
print("  PASS: unbiased at every tested strength.")

print("\n=== Check 4: variance comparison, state-dependent tilt vs constant-shift IS ===")
best_const_shift, _ = calibrate_drift_shift(S0, K, B, r, sigma, T, n_steps, r_state=777,
                                             use_bridge=True, q=q, pilot_n=20_000)
rng = np.random.default_rng(10)
_, _, var_const, _ = price_barrier_importance_sampling(
    S0, K, B, r, sigma, T, n_steps, n_draws=100_000, rng=rng,
    drift_shift=best_const_shift, use_bridge=True, q=q)

print(f"  Constant-shift IS (shift={best_const_shift:+.2f}): variance = {var_const:.4f}")
for strength in [0.3, 0.6, 1.0]:
    rng = np.random.default_rng(10)
    _, _, var_sd, _ = price_barrier_state_dependent_is(
        S0, K, B, r, sigma, T, n_steps, n_draws=100_000, rng=rng, q=q, strength=strength)
    print(f"  State-dependent tilt (strength={strength:.1f}):    variance = {var_sd:.4f}  "
          f"({'better' if var_sd < var_const else 'worse'} than constant-shift)")

print("\nAll checks complete.")


=== Check 1: no-touch closed form vs high-N survival-probability MC ===
  Closed form:        0.41291
  Bridge MC (N=500k): 0.41204 +/- 0.00129
  Difference: 0.00087  (within 3 std errors)
  PASS

=== Check 2: survival-based control variate is unbiased, and its actual correlation ===
  Closed-form call price:         0.2676
  Survival-CV estimate:           0.2767 +/- 0.0077
  Agrees with closed form: True
  Empirical correlation(Y, survival): 0.2028
  (this is the number that actually matters -- not assumed, measured)

=== Check 3: state-dependent tilt is unbiased, at several 'strength' settings ===
  Baseline (plain bridge, N=200k): price=0.2674 +/- 0.0055
  strength=0.0: price=0.2634 +/- 0.0076  var=1.5168  unbiased=True
  strength=0.3: price=0.2666 +/- 0.0049  var=0.6189  unbiased=True
  strength=0.6: price=0.2681 +/- 0.0038  var=0.3800  unbiased=True
  strength=1.0: price=0.2686 +/- 0.0042  var=0.4617  unbiased=True
  PASS: unbiased at every tested strength.

=== Check 4: variance

In [11]:
"""
Compares ALL barrier techniques attempted in this project, at the same
fixed compute budget, so the improvement from the domain-specific fixes
is visible against the naive baseline from run_experiment.py:

  1. Naive discrete indicator + continuity correction (original approach)
  2. Naive discrete indicator + vanilla-call control variate (original approach)
  3. Brownian bridge conditional expectation (Rao-Blackwellization)
  4. Brownian bridge + calibrated importance sampling

Uses market_params.json if present, else the same synthetic defaults
used elsewhere in this project.
"""
import os
import json
import time
import numpy as np
import pandas as pd


DEFAULT_PARAMS = dict(S0=100.0, K=100.0, r=0.03, q=0.0, sigma=0.25, T=1.0, n_steps=52)
BARRIER_MULTIPLIER = 1.15
PARAMS_FILE = "market_params.json"
N_TOTAL_DRAWS = 200_000
N_REPS = 20
RNG_SEED = 12345


def load_params():
    if os.path.exists(PARAMS_FILE):
        with open(PARAMS_FILE) as f:
            data = json.load(f)
        S0, r, sigma, T = data["S0"], data["r"], data["sigma"], data["T_years"]
        q = data.get("q", 0.0)
        label = f"{data['ticker']}, {data['fetched_at'][:10]}"
        print(f"Loaded CALIBRATED market params: {label}")
        return dict(S0=S0, K=S0, r=r, q=q, sigma=sigma, T=T, n_steps=52), label
    print(f"{PARAMS_FILE} not found -- using SYNTHETIC placeholder params.")
    return dict(DEFAULT_PARAMS, K=DEFAULT_PARAMS["S0"]), "synthetic placeholder values"


def repeated_run(fn, n_reps, **kwargs):
    prices, variances, times = [], [], []
    for rep in range(n_reps):
        rng = np.random.default_rng(RNG_SEED + rep)
        t0 = time.perf_counter()
        price, se, var, n_paths = fn(rng=rng, **kwargs)
        elapsed = time.perf_counter() - t0
        prices.append(price)
        variances.append(var)
        times.append(elapsed)
    return np.mean(prices), np.mean(variances), np.mean(times)


if __name__ == "__main__":
    params, source_label = load_params()
    S0, K, r, q, sigma, T, n_steps = (
        params["S0"], params["K"], params["r"], params["q"],
        params["sigma"], params["T"], params["n_steps"]
    )
    B = S0 * BARRIER_MULTIPLIER

    print(f"\nUP-AND-OUT BARRIER CALL comparison  (S0={S0:.2f}, K={K:.2f}, B={B:.2f}, "
          f"r={r*100:.2f}%, q={q*100:.2f}%, sigma={sigma*100:.2f}%, T={T:.2f}y)  [{source_label}]\n")

    # 1. Naive + continuity correction (from run_experiment.py's "plain")
    price1, var1, time1 = repeated_run(
        price_barrier_mc, N_REPS, S0=S0, K=K, B=B, r=r, sigma=sigma, T=T, n_steps=n_steps,
        n_draws=N_TOTAL_DRAWS, method="plain", continuity_correction=True, q=q)

    # 2. Naive + vanilla-call control variate (from run_experiment.py's "control")
    price2, var2, time2 = repeated_run(
        price_barrier_mc, N_REPS, S0=S0, K=K, B=B, r=r, sigma=sigma, T=T, n_steps=n_steps,
        n_draws=N_TOTAL_DRAWS, method="control", continuity_correction=True, q=q)

    # 3. Brownian bridge conditional expectation (all four method variants,
    #    since the vanilla-call control variate may behave very differently
    #    now that it's being applied to a smooth payoff instead of a
    #    discontinuous one)
    price3a, var3a, time3a = repeated_run(
        price_barrier_bridge_mc, N_REPS, S0=S0, K=K, B=B, r=r, sigma=sigma, T=T, n_steps=n_steps,
        n_draws=N_TOTAL_DRAWS, method="plain", q=q)
    price3b, var3b, time3b = repeated_run(
        price_barrier_bridge_mc, N_REPS, S0=S0, K=K, B=B, r=r, sigma=sigma, T=T, n_steps=n_steps,
        n_draws=N_TOTAL_DRAWS, method="antithetic", q=q)
    price3c, var3c, time3c = repeated_run(
        price_barrier_bridge_mc, N_REPS, S0=S0, K=K, B=B, r=r, sigma=sigma, T=T, n_steps=n_steps,
        n_draws=N_TOTAL_DRAWS, method="control", q=q)
    price3d, var3d, time3d = repeated_run(
        price_barrier_bridge_mc, N_REPS, S0=S0, K=K, B=B, r=r, sigma=sigma, T=T, n_steps=n_steps,
        n_draws=N_TOTAL_DRAWS, method="antithetic_control", q=q)

    # 4. Brownian bridge + calibrated importance sampling
    best_shift, _ = calibrate_drift_shift(S0, K, B, r, sigma, T, n_steps, r_state=777,
                                           use_bridge=True, q=q, pilot_n=20_000)
    print(f"Calibrated drift shift for importance sampling: {best_shift:+.2f}\n")

    def bridge_is_fn(rng, **kw):
        return price_barrier_importance_sampling(drift_shift=best_shift, use_bridge=True,
                                                   rng=rng, **kw)

    price4, var4, time4 = repeated_run(
        bridge_is_fn, N_REPS, S0=S0, K=K, B=B, r=r, sigma=sigma, T=T, n_steps=n_steps,
        n_draws=N_TOTAL_DRAWS, q=q)

    # 5. Survival-based control variate (on top of bridge, no IS)
    def survival_cv_fn(rng, **kw):
        price, se, var, n, corr = price_barrier_bridge_survival_cv(rng=rng, **kw)
        return price, se, var, n

    price5, var5, time5 = repeated_run(
        survival_cv_fn, N_REPS, S0=S0, K=K, B=B, r=r, sigma=sigma, T=T, n_steps=n_steps,
        n_draws=N_TOTAL_DRAWS, q=q)

    rng_corr = np.random.default_rng(RNG_SEED)
    _, _, _, _, measured_corr = price_barrier_bridge_survival_cv(
        S0=S0, K=K, B=B, r=r, sigma=sigma, T=T, n_steps=n_steps, n_draws=N_TOTAL_DRAWS, rng=rng_corr, q=q)

    # 6. State-dependent (corridor-pinning) tilt -- calibrated strength
    best_strength, strength_sweep = calibrate_tilt_strength(
        S0, K, B, r, sigma, T, n_steps, r_state=888, q=q, pilot_n=20_000)
    print(f"Calibrated pinning strength for state-dependent tilt: {best_strength:.2f}\n")

    def state_dependent_fn(rng, **kw):
        return price_barrier_state_dependent_is(strength=best_strength, use_bridge=True, rng=rng, **kw)

    price6, var6, time6 = repeated_run(
        state_dependent_fn, N_REPS, S0=S0, K=K, B=B, r=r, sigma=sigma, T=T, n_steps=n_steps,
        n_draws=N_TOTAL_DRAWS, q=q)

    rows = [
        dict(method="1. Naive discrete + continuity correction (original)",
             price=round(price1, 4), variance=round(var1, 5),
             variance_reduction_pct=0.0, time_sec=round(time1, 3)),
        dict(method="2. Naive discrete + vanilla-call control variate (original)",
             price=round(price2, 4), variance=round(var2, 5),
             variance_reduction_pct=round(100 * (1 - var2 / var1), 2), time_sec=round(time2, 3)),
        dict(method="3a. Brownian bridge, plain",
             price=round(price3a, 4), variance=round(var3a, 5),
             variance_reduction_pct=round(100 * (1 - var3a / var1), 2), time_sec=round(time3a, 3)),
        dict(method="3b. Brownian bridge + antithetic",
             price=round(price3b, 4), variance=round(var3b, 5),
             variance_reduction_pct=round(100 * (1 - var3b / var1), 2), time_sec=round(time3b, 3)),
        dict(method="3c. Brownian bridge + vanilla-call control variate",
             price=round(price3c, 4), variance=round(var3c, 5),
             variance_reduction_pct=round(100 * (1 - var3c / var1), 2), time_sec=round(time3c, 3)),
        dict(method="3d. Brownian bridge + antithetic + control variate",
             price=round(price3d, 4), variance=round(var3d, 5),
             variance_reduction_pct=round(100 * (1 - var3d / var1), 2), time_sec=round(time3d, 3)),
        dict(method=f"4. Brownian bridge + importance sampling (shift={best_shift:+.2f})",
             price=round(price4, 4), variance=round(var4, 5),
             variance_reduction_pct=round(100 * (1 - var4 / var1), 2), time_sec=round(time4, 3)),
        dict(method=f"5. Brownian bridge + survival-based control variate (corr={measured_corr:.2f})",
             price=round(price5, 4), variance=round(var5, 5),
             variance_reduction_pct=round(100 * (1 - var5 / var1), 2), time_sec=round(time5, 3)),
        dict(method=f"6. Brownian bridge + state-dependent corridor-pinning tilt (strength={best_strength:.2f})",
             price=round(price6, 4), variance=round(var6, 5),
             variance_reduction_pct=round(100 * (1 - var6 / var1), 2), time_sec=round(time6, 3)),
    ]
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    df.to_csv("barrier_advanced_results.csv", index=False)
    print("\nSaved barrier_advanced_results.csv")


Loaded CALIBRATED market params: SPY, 2026-09-20

UP-AND-OUT BARRIER CALL comparison  (S0=761.69, K=761.69, B=875.94, r=4.12%, q=0.98%, sigma=23.99%, T=0.99y)  [SPY, 2026-09-20]

Calibrated drift shift for importance sampling: +0.10

Calibrated pinning strength for state-dependent tilt: 0.80

                                                                    method  price  variance  variance_reduction_pct  time_sec
                      1. Naive discrete + continuity correction (original) 2.1850 109.68769                    0.00     0.374
               2. Naive discrete + vanilla-call control variate (original) 2.1853 109.42226                    0.24     0.311
                                                3a. Brownian bridge, plain 2.2818 100.46755                    8.41     0.776
                                          3b. Brownian bridge + antithetic 2.2853 100.65376                    8.24     1.351
                        3c. Brownian bridge + vanilla-call control variate 2